# Pre-annotation du corpus — Guide Universitaire (Colab GPU)

Variante Colab de `app/nlp/pre_annoter_corpus.py` : meme logique (echantillon equilibre par categorie, puis NER + classification zero-shot pour pre-remplir des suggestions a corriger a la main), mais executee ici avec acceleration GPU plutot qu'en local sur CPU.

**Pourquoi ce notebook et pas le script local ?** Le pipeline zero-shot (`MoritzLaurer/mDeBERTa-v3-base-mnli-xnli`) fait deux classifications par offre (secteur + categorie de contrat) en plus du NER : sur CPU, quelques centaines d'offres peuvent prendre plusieurs heures. Un GPU Colab gratuit (T4) ramene ca a quelques minutes — le meme arbitrage que pour l'entrainement (`entrainement_colab.ipynb`).

**Ce que ce notebook ne fait toujours pas :** produire une verite terrain. Les colonnes `secteur_zero_shot` / `categorie_contrat_zero_shot` restent des suggestions a corriger par un humain (`secteur_final` / `categorie_contrat_finale` / `entites_finales_bio` sont exportees vides), exactement comme dans le script local.

**Avant de lancer :** dans Colab, aller dans `Execution > Modifier le type d'execution` et choisir **GPU** (T4 suffit).

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else -1
print(f"Device disponible : {'GPU' if device == 0 else 'CPU'}")
if device == -1:
    print("ATTENTION : pas de GPU detecte. Executer > Modifier le type d'execution > GPU.")

## 1. Uploader le corpus collecte en local

Uploade ici le CSV produit par `python -m app.nlp.collecte_corpus` (ex. `corpus_offres_burkina_*.csv`), avec ses colonnes `secteur_suggere` / `confiance_suggestion` deja remplies par `app/nlp/mapping_secteurs.py`.

In [ ]:
from google.colab import files
import io
import csv

fichiers_uploades = files.upload()
nom_fichier_corpus = next(iter(fichiers_uploades))
contenu_corpus = fichiers_uploades[nom_fichier_corpus].decode("utf-8-sig")
toutes_les_lignes = list(csv.DictReader(io.StringIO(contenu_corpus)))
print(f"{len(toutes_les_lignes)} lignes chargees depuis {nom_fichier_corpus}")

## 2. Echantillonnage equilibre

Memes constantes et meme fonction que `pre_annoter_corpus.py` : plutot que de pre-annoter tout le corpus (inutile — un echantillon annote suffit, et ca reproduirait le desequilibre du corpus brut, voir RAPPORT_PROJET.md section 5.6), un nombre maximum d'offres par categorie suggeree est preleve, en priorisant les suggestions de confiance "haute".

In [ ]:
from collections import defaultdict

SECTEURS_CIBLES = [
    "Informatique", "BTP_Genie_Civil", "Comptabilite_Finance", "Mines_Industrie",
    "Agriculture", "ONG_Social", "Microfinance_Banque", "Marketing_Commercial",
    "Juridique", "Ressources_Humaines", "Education", "Sante", "Logistique",
]
CATEGORIES_CONTRAT_CIBLES = ["CDI", "CDD", "Stage", "Alternance", "Freelance", "Temps_Partiel"]

MAX_PAR_CATEGORIE = 25  # modifiable directement ici (equivalent de --max-par-categorie en local)
LONGUEUR_MAX_TEXTE = 2000  # troncature raisonnable pour le zero-shot

COLONNES_SORTIE = [
    "id_externe", "titre_poste", "nom_entreprise", "localisation", "description",
    "source", "mots_cles_recherche",
    "secteur_suggere", "confiance_suggestion",
    "secteur_zero_shot", "score_secteur_zero_shot", "accord_secteur",
    "categorie_contrat_zero_shot", "score_categorie_zero_shot",
    "entites_detectees",
    "secteur_final", "categorie_contrat_finale", "entites_finales_bio",
]


def selectionner_echantillon_equilibre(lignes, max_par_categorie):
    par_categorie = defaultdict(list)
    for ligne in lignes:
        cle = ligne.get("secteur_suggere") or "(non_categorise)"
        par_categorie[cle].append(ligne)

    echantillon = []
    for cle, groupe in par_categorie.items():
        groupe_trie = sorted(groupe, key=lambda l: l.get("confiance_suggestion") != "haute")
        echantillon.extend(groupe_trie[:max_par_categorie])

    return echantillon


echantillon = selectionner_echantillon_equilibre(toutes_les_lignes, MAX_PAR_CATEGORIE)
print(f"Echantillon equilibre : {len(echantillon)} offres sur {len(toutes_les_lignes)} (max {MAX_PAR_CATEGORIE}/categorie)")

## 3. Chargement des pipelines (GPU)

In [ ]:
from transformers import pipeline as hf_pipeline

ner_pipeline = hf_pipeline(
    task="ner", model="Jean-Baptiste/camembert-ner", aggregation_strategy="simple", device=device
)
classification_pipeline = hf_pipeline(
    task="zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device=device
)

## 4. Pre-annotation

Pour chaque offre de l'echantillon : classification zero-shot du secteur (13 candidats) et de la categorie de contrat (6 candidats), puis NER, exactement comme `pre_annoter_corpus.py`.

In [ ]:
from tqdm.auto import tqdm

resultats = []
for ligne in tqdm(echantillon, desc="Pre-annotation"):
    texte = f"{ligne['titre_poste']}. {ligne['description']}"[:LONGUEUR_MAX_TEXTE]

    secteur_zs = classification_pipeline(texte, candidate_labels=SECTEURS_CIBLES)
    categorie_zs = classification_pipeline(texte, candidate_labels=CATEGORIES_CONTRAT_CIBLES)
    entites = ner_pipeline(texte)

    secteur_zero_shot = secteur_zs["labels"][0]
    secteur_mapping = ligne.get("secteur_suggere", "")
    accord = "n/a" if not secteur_mapping else ("oui" if secteur_mapping == secteur_zero_shot else "non")

    ligne.update({
        "secteur_zero_shot": secteur_zero_shot,
        "score_secteur_zero_shot": round(float(secteur_zs["scores"][0]), 3),
        "accord_secteur": accord,
        "categorie_contrat_zero_shot": categorie_zs["labels"][0],
        "score_categorie_zero_shot": round(float(categorie_zs["scores"][0]), 3),
        "entites_detectees": " | ".join(f"{e['entity_group']}:{e['word']}" for e in entites),
        "secteur_final": "",
        "categorie_contrat_finale": "",
        "entites_finales_bio": "",
    })
    resultats.append(ligne)

print(f"{len(resultats)} offres pre-annotees.")

## 5. Export et telechargement

Meme nommage que le script local (`<corpus>_pre_annote.csv`), telecharge directement depuis Colab pour reprendre l'annotation manuelle en local.

In [ ]:
import os

nom_sortie = os.path.splitext(nom_fichier_corpus)[0] + "_pre_annote.csv"
with open(nom_sortie, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.DictWriter(f, fieldnames=COLONNES_SORTIE)
    writer.writeheader()
    writer.writerows(resultats)
print(f"{len(resultats)} offres pre-annotees exportees dans {nom_sortie}")

files.download(nom_sortie)